[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/04_capacity_prevalence_queues.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Capacity, prevalence and daily queues

Allow about 30 minutes. Can a policy that matches the review team's capacity on
average still lead to overload? You will need to project the forest's development
operating points to a planning prevalence, select a cutoff under a tolerance for
daily overflow, compare a top-k cap on simulated days, and determine how far
prevalence can drift before the selected cutoff fails the tolerance.

The action is **send this recording for AF review**, priced in reviews as per the
core notebook: $C_{TP}$ = 1, $C_{FP}$ = 1, $C_{FN}$ = 5, and $C_{TN}$ = 0. Please
attempt this on your own. Replace each `...` with your solution and run the checker
below it. If you are stuck for about 10 minutes, use the stuck switch below the
problem (if available) or view the worked copy in `extension/solutions/`. Note that
a later cell that needs an unfinished problem prints `Waiting for ...`: once you've
completed that problem or used its stuck switch, you can click in the waiting cell,
select **Runtime > Run before** and then run the waiting cell.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

from clinical_utility import data, deployment, reference
from clinical_utility import extension_checks as check, extension_plots as view

costs = dict(reference.DEFAULT_COSTS)
capacity = 50            # reviews per day
arrivals_per_day = 1000  # recordings per day
prevalence = 0.05        # the planning prevalence
tolerance = 0.10         # the largest acceptable chance that a day's demand exceeds capacity
scores = data.load_scores()
development = scores[scores["split"] == "development"]
points = reference.operating_points(development["af"], development["random_forest"], costs)

## Project the operating points to the service

The review team has a capacity of 50 recordings a day, however, 1,000 recordings
arrive daily. The service expects AF in 5% of all recordings instead of the 9.2% we
saw in the development split. We need to project each cutoff to this new prevalence
while keeping its sensitivity and specificity fixed: the AF group is
`per * prevalence` and the non-AF group is its complement. The AF group will have
recordings flagged at the sensitivity and the non-AF group at the false-positive
rate, one minus the specificity. For example, with a sensitivity of 80%, a
specificity of 90% and a prevalence of 5%, we would expect the following for 1,000
recordings: TP 40, FN 10, FP 95 and TN 855. Note that PPV does not carry over, as it
changes when the sizes of the two groups change.

### Problem A: Make

Write `project(points, prevalence, costs, per=1000)`, a function that takes in an
operating-point table with `threshold`, `sensitivity` and `specificity` columns and
produces a DataFrame with the `threshold`, the expected number of `TP`, `FN`, `FP`
and `TN` per `per` recordings as well as the expected number of `reviews` (TP + FP),
`af_missed` (FN), and `loss` (all four counts priced). Ensure that the values are
not rounded. This should be solved in a vectorised manner using pandas arithmetic
without loops.

In [ ]:
def project(points, prevalence, costs, per=1000):
    return ...
_ = check.run("queue_project", project)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# project = check.use_reference("project")

In [ ]:
if check.ready(project=project):
    projected = project(points, prevalence, costs)
    unconstrained = reference.cheapest(projected)
    mean_feasible = reference.cheapest(projected[projected["reviews"] <= capacity])
    display(pd.DataFrame({"cheapest overall": unconstrained,
                          "cheapest with mean reviews within capacity": mean_feasible}).loc[
        ["threshold", "reviews", "TP", "af_missed", "loss"]].round(3))

The overall cheapest policy would result in 72.5 reviews a day, which is above the
capacity of the team. If we limit the *mean* to the team's capacity, the cutoff
would be 0.467, which would result in 49.7 reviews a day but would miss 19.8 AF
recordings a day compared to 13.3. Notice that a capacity is a constraint and not a
higher price for a review.

## A mean within capacity is not a day within capacity

Daily arrivals fluctuate, and so too will the number flagged. If we assume that the
number of daily arrivals is Poisson and that each recording is independently flagged
for review, we can model the number of daily reviews as a Poisson distribution with
mean given by the policy. We can then calculate with
`stats.poisson.sf(capacity, mean)` the probability that a given day's demand will
exceed capacity (strictly more than 50).

### Problem B: Make

Write `capacity_policy(projected, capacity, tolerance)`. Given a tolerance for daily
overflow, append an `overflow` column with the probability that a given day will
exceed capacity, filter for only policies that have an overflow less than or equal
to the tolerance, and then return the cheapest one (a pandas Series; fewest reviews
on a tie). The overflow tolerance is a service-level preference and not a
statistical result.

In [ ]:
def capacity_policy(projected, capacity, tolerance):
    return ...
_ = check.run("queue_capacity", capacity_policy)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# capacity_policy = check.use_reference("capacity_policy")

In [ ]:
if check.ready(project=project, capacity_policy=capacity_policy):
    projected = project(points, prevalence, costs)
    mean_feasible = reference.cheapest(projected[projected["reviews"] <= capacity])
    chosen = capacity_policy(projected, capacity, tolerance)
    days = deployment.simulate_days(arrivals_per_day, mean_feasible["reviews"] / arrivals_per_day)
    print("Mean-feasible policy: Poisson chance of a day over capacity",
          round(stats.poisson.sf(capacity, mean_feasible["reviews"]), 3),
          "; simulated share of 365 days over capacity", round(np.mean(days > capacity), 3))
    display(view.queue(days, capacity))
    display(chosen[["threshold", "reviews", "TP", "af_missed", "loss", "overflow"]].round(3))

If we read the histogram as a year of days for the policy that is feasible on
average, its mean fits, but it has a 44% Poisson probability of exceeding capacity
on any given day, and in our seeded simulation 47% of days exceeded the capacity,
with the busiest day requiring 71 reviews. Note that the simulation draws a Poisson
number of arrivals and then a binomial number of reviews. This does not account for
weekday effects, batching of referrals, or the carry-over of queues from the prior
day. If we apply a 10% overflow tolerance, the cheapest policy has a cutoff of
0.543. This has a daily review rate of 41.4 and misses 22.7 AF recordings per day.
The additional missed recordings are the price of a policy that can consistently
deliver daily within capacity.

## A cap instead of a cutoff

We could instead use a top-k policy and only review the 50 highest scores each day,
whatever they are. This would mean that we would never exceed the capacity, however,
the recordings it drops are the ones a fixed cutoff would have reviewed. The cell
below simulates what this would look like for a year of daily batches at the
planning prevalence. This simulation assumes that each day a Poisson number of
recordings arrive and that a binomial number have AF. We then resample the
development forest's scores for AF and non-AF recordings to get the scores for each
recording. We then compare the cap with the cheapest overall cutoff (0.285).

In [ ]:
if check.ready(project=project):
    projected = project(points, prevalence, costs)
    unconstrained = reference.cheapest(projected)
    y = development["af"].to_numpy()
    batches = deployment.top_k_days(development["random_forest"][y], development["random_forest"][~y],
                                    prevalence, arrivals_per_day, capacity, unconstrained["threshold"])
    means = batches.mean()
    display(pd.DataFrame({
        "fixed cutoff": {"reviews a day": means["flagged"], "AF detected a day": means["cutoff_detected"],
                         "AF missed a day": means["cutoff_missed"],
                         "net benefit": means["cutoff_net_benefit"], "deferred a day": 0.0},
        "top-k cap": {"reviews a day": means["reviewed"], "AF detected a day": means["cap_detected"],
                      "AF missed a day": means["cap_missed"], "net benefit": means["cap_net_benefit"],
                      "deferred a day": means["deferred"]}}).round(3))
    print("AF recordings deferred a day under the cap: mean", round(means["af_deferred"], 1),
          "; worst day", batches["af_deferred"].max())

On an average simulated day the fixed cutoff flags 72.5 recordings, while the cap
reviews 50 and defers 22.5, of which 6.7 are AF recordings. Counting every AF
recording not reviewed that day, the cap would detect 30.1 and miss 20.2 AF
recordings each day compared to 36.7 and 13.5 for the fixed cutoff. The cap would
have a net benefit of 0.0251 compared to 0.0278 for the fixed cutoff. Deferral
happens on 99% of days, and the maximum number of deferred AF recordings on a single
day is 18. We can no longer simply carry over the sensitivity and specificity of a
fixed cutoff to the cap, as its effective cutoff changes each day based on the
scores. We also now need a stated pathway for the deferred recordings: do they get
reviewed the next day, do we bring in extra staff to review them, or do they just
wait longer? Each has its own consequences.

## How far can prevalence drift?

### Problem C: Make

Fix the cutoff from Problem B and its development sensitivity and specificity. Using
`project` for that single operating point, find the highest prevalence from 1% to
10% in increments of 0.5% that still keeps the overflow probability within the
tolerance. Store the highest prevalence as a fraction in `highest_safe_prevalence`.

In [ ]:
highest_safe_prevalence = ...
_ = check.run("queue_prevalence", highest_safe_prevalence)

<details><summary>Open after your own attempt</summary>

#### Little headroom

The result is 5.0%, which is the highest safe value on the grid. This is the same as
the planning prevalence for the service. If we solve for the boundary instead of
stepping through the grid, we get about 5.14%. The cutoff we selected is the
cheapest that meets the tolerance at 5%, and so it has minimal headroom for
prevalence drift. The cutoff that is feasible on average breaks the tolerance above
about 3.70% (with the last safe grid value at 3.5%). If we expect prevalence drift,
we should monitor the prevalence and daily volumes and select a cutoff that has
headroom or a stated overflow pathway.

## Check your report

- Determine the number of recordings with and without AF before projecting counts.
- Distinguish a constraint on the average from a tolerance for overflow days.
- What would the top-k cap do to a recording on a busy day?
- What quantities would you monitor post-deployment?

**Question to consider:** what assumptions break if demand is clustered and
yesterday's queue carries over? What data would you want to collect from the service
before selecting a new policy?

</details>